In [46]:
import pandas as pd
import numpy as np
from datetime import datetime

def parse_aqi_file(filepath, station_name):
    
    with open(filepath, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f.readlines()]
    
    records = []
    current_month = None
    
    for line in lines:
        
        # skip empty separators
        if line in ['""', '']:
            continue
        
        # detect month header
        if "-" in line and "00:00:00" in line:
            
            current_month = line.split(",")[0]
            continue
        
        parts = line.split(",")
        
        # day rows
        if parts[0].isdigit():
            
            day = int(parts[0])
            
            if current_month is None:
                continue
            
            values = []
            
            for v in parts[1:]:
                
                if v.strip() != "":
                    try:
                        values.append(float(v))
                    except:
                        pass
            
            if len(values) == 0:
                continue
            
            daily_avg = np.mean(values)
            
            month_dt = datetime.strptime(
                current_month,
                "%B-%Y"
            )
            
            full_date = datetime(
                month_dt.year,
                month_dt.month,
                day
            )
            
            records.append({
                "date": full_date,
                "station": station_name,
                "daily_avg_aqi": round(daily_avg, 2)
            })
    
    return pd.DataFrame(records)

In [50]:
station_files = {
    "Anand Vihar": "../data/raw/aqi/anand_vihar.csv",
    "Ashok Vihar": "../data/raw/aqi/ashok_vihar.csv",
    "Chandni Chowk": "../data/raw/aqi/chandni_chowk.csv",
    "Mundka": "../data/raw/aqi/mundka.csv",
    "Najafgarh": "../data/raw/aqi/najafgarh.csv"
}

all_dfs = []

for station, path in station_files.items():

    df = parse_aqi_file(path, station)

    print(
        station,
        len(df),
        df["date"].min(),
        df["date"].max()
    )

    all_dfs.append(df)

aqi_df = pd.concat(all_dfs, ignore_index=True)

print(aqi_df.shape)

Anand Vihar 2131 2017-10-17 00:00:00 2023-12-31 00:00:00
Ashok Vihar 2141 2018-02-01 00:00:00 2023-12-31 00:00:00
Chandni Chowk 761 2020-11-08 00:00:00 2023-06-08 00:00:00
Mundka 1984 2018-07-01 00:00:00 2023-12-31 00:00:00
Najafgarh 2090 2018-02-01 00:00:00 2023-12-31 00:00:00
(9107, 3)


In [ ]:
mapping = pd.read_csv(
    "../data/processed/station_district_mapping.csv"
)

mapping



In [55]:
aqi_df = aqi_df.merge(
    mapping,
    on="station",
    how="left"
)

aqi_df.head()

,date,station,daily_avg_aqi,district_x,district_y,year,district
0,2017-10-17,Anand Vihar,183.30,East,East,2017,East
1,2017-10-18,Anand Vihar,130.56,East,East,2017,East
2,2017-10-19,Anand Vihar,111.58,East,East,2017,East
3,2017-10-20,Anand Vihar,152.83,East,East,2017,East
4,2017-10-21,Anand Vihar,146.19,East,East,2017,East


In [56]:
aqi_df["year"] = aqi_df["date"].dt.year


In [57]:
district_aqi = (
    aqi_df
    .groupby(
        ["district", "year"]
    )["daily_avg_aqi"]
    .mean()
    .reset_index()
)

district_aqi.rename(
    columns={
        "daily_avg_aqi":"avg_aqi"
    },
    inplace=True
)

district_aqi

,district,year,avg_aqi
0,Central,2020,338.223000
1,Central,2021,249.487528
2,Central,2022,243.101131
3,Central,2023,196.146940
4,East,2017,375.926222
5,East,2018,294.125559
6,East,2019,251.353102
7,East,2020,212.792824
8,East,2021,251.850000
9,East,2022,276.971041


In [58]:
district_aqi.to_csv(
    "../data/processed/district_aqi.csv",
    index=False
)